In [ ]:
!pip install -q ucimlrepo

import os
import json
import warnings
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/audit", exist_ok=True)

print("Project directories created.")

print("=" * 60)
print("DATA ACQUISITION")
print("=" * 60)

heart_disease = fetch_ucirepo(id=45)

X_raw = heart_disease.data.features.copy()
y_raw = heart_disease.data.targets.copy()

print("Dataset successfully acquired.")
print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

source_info = {
    "dataset": "UCI Heart Disease",
    "UCI_dataset_id": 45,
    "source": "UCI Machine Learning Repository",
    "task": "Heart Disease Classification",
    "target_original": "num",
    "target_transformation": "num > 0 -> 1, num = 0 -> 0",
    "random_state": RANDOM_STATE
}

with open("data/audit/source_info.json", "w") as f:
    json.dump(source_info, f, indent=4)

print("\nSource information saved.")

df = X_raw.copy()
df["num"] = y_raw["num"].values

print("\nInitial dataset shape:", df.shape)
print(df.head())

print("\n" + "=" * 60)
print("SCHEMA VALIDATION")
print("=" * 60)

expected_features = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal"
]

expected_columns = expected_features + ["num"]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

unexpected_columns = [
    col for col in df.columns
    if col not in expected_columns
]

if missing_columns:
    raise ValueError(
        f"Schema validation failed. Missing columns: {missing_columns}"
    )

if unexpected_columns:
    print("Warning: Unexpected columns:", unexpected_columns)

print("Schema validation PASSED.")
print("Expected columns found:", len(expected_columns))

print("\nData types:")

dtype_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isnull().sum().values,
    "unique_values": df.nunique().values
})

print(dtype_audit)

dtype_audit.to_csv(
    "data/audit/schema_audit.csv",
    index=False
)

print("\nSchema audit saved.")

audit = []

for column in df.columns:

    audit.append({
        "column": column,
        "dtype": str(df[column].dtype),
        "rows": len(df),
        "missing_count": int(df[column].isnull().sum()),
        "missing_percentage": round(
            df[column].isnull().mean() * 100, 2
        ),
        "unique_values": int(df[column].nunique()),
        "duplicate_count": int(df[column].duplicated().sum())
    })

quality_audit = pd.DataFrame(audit)

print("\nInitial Data Quality Audit:")
display(quality_audit)

quality_audit.to_csv(
    "data/audit/data_quality_before_cleaning.csv",
    index=False
)

print("\n" + "=" * 60)
print("DUPLICATE RECORD CHECK")
print("=" * 60)

duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

if duplicate_count > 0:

    df = df.drop_duplicates().reset_index(drop=True)

    print(
        f"Removed {duplicate_count} duplicate records."
    )

else:
    print("No duplicate records found.")

print("\n" + "=" * 60)
print("INVALID RECORD VALIDATION")
print("=" * 60)

invalid_counts = {}

invalid_counts["age"] = (df["age"] <= 0).sum()

invalid_counts["trestbps"] = (df["trestbps"] <= 0).sum()

invalid_counts["chol"] = (df["chol"] <= 0).sum()

invalid_counts["thalach"] = (df["thalach"] <= 0).sum()

invalid_counts["num"] = (
    ~df["num"].isin([0, 1, 2, 3, 4])
).sum()

invalid_audit = pd.DataFrame(
    list(invalid_counts.items()),
    columns=["column", "invalid_count"]
)

print(invalid_audit)

invalid_audit.to_csv(
    "data/audit/invalid_records.csv",
    index=False
)

before_invalid_removal = len(df)

df = df[
    (df["age"] > 0) &
    (df["trestbps"] > 0) &
    (df["chol"] > 0) &
    (df["thalach"] > 0) &
    (df["num"].isin([0, 1, 2, 3, 4]))
].copy()

after_invalid_removal = len(df)

print(
    "Invalid records removed:",
    before_invalid_removal - after_invalid_removal
)

print("\n" + "=" * 60)
print("TARGET TRANSFORMATION")
print("=" * 60)

df["target"] = (df["num"] > 0).astype(int)

df.drop(columns=["num"], inplace=True)

print("\nBinary target distribution:")
print(df["target"].value_counts())

print("\nTarget percentages:")
print(
    df["target"].value_counts(normalize=True).mul(100).round(2)
)

X = df.drop(columns=["target"])
y = df["target"]

print("\nFeatures:", X.shape)
print("Target:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print(
    "\nTraining target distribution:\n",
    y_train.value_counts(normalize=True)
)

print(
    "\nTesting target distribution:\n",
    y_test.value_counts(normalize=True)
)

print("\n" + "=" * 60)
print("OUTLIER DETECTION")
print("=" * 60)

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

outlier_report = []

for column in numeric_features:

    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    train_outliers = (
        (X_train[column] < lower) |
        (X_train[column] > upper)
    ).sum()

    test_outliers = (
        (X_test[column] < lower) |
        (X_test[column] > upper)
    ).sum()

    outlier_report.append({
        "feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower,
        "upper_bound": upper,
        "train_outliers": train_outliers,
        "test_outliers": test_outliers
    })

outlier_report = pd.DataFrame(outlier_report)

display(outlier_report)

outlier_report.to_csv(
    "data/audit/outlier_report.csv",
    index=False
)

class IQRClipper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):

        X_df = pd.DataFrame(X)

        self.lower_bounds_ = (
            X_df.quantile(0.25)
            - self.factor *
            (X_df.quantile(0.75) - X_df.quantile(0.25))
        )

        self.upper_bounds_ = (
            X_df.quantile(0.75)
            + self.factor *
            (X_df.quantile(0.75) - X_df.quantile(0.25))
        )

        return self

    def transform(self, X):

        X_df = pd.DataFrame(X).copy()

        for column in X_df.columns:

            X_df[column] = X_df[column].clip(
                lower=self.lower_bounds_[column],
                upper=self.upper_bounds_[column]
            )

        return X_df.values

print("\n" + "=" * 60)
print("BUILDING PREPROCESSING PIPELINE")
print("=" * 60)

preprocessing_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "outlier_clipper",
        IQRClipper(factor=1.5)
    ),

    (
        "scaler",
        StandardScaler()
    )
])

print("Fitting preprocessing pipeline on TRAINING data...")

X_train_processed = preprocessing_pipeline.fit_transform(
    X_train
)

print("Training preprocessing complete.")

print("Transforming TEST data using training parameters...")

X_test_processed = preprocessing_pipeline.transform(
    X_test
)

print("Testing preprocessing complete.")

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=X_train.columns,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=X_test.columns,
    index=X_test.index
)

print("\n" + "=" * 60)
print("POST-PROCESSING QUALITY CHECK")
print("=" * 60)

print(
    "Training missing values:",
    X_train_processed.isnull().sum().sum()
)

print(
    "Testing missing values:",
    X_test_processed.isnull().sum().sum()
)

assert X_train_processed.isnull().sum().sum() == 0
assert X_test_processed.isnull().sum().sum() == 0

print("Missing-value validation PASSED.")

print("\n" + "=" * 60)
print("DATA DRIFT CHECK")
print("=" * 60)

def calculate_psi(expected, actual, bins=10):

    expected = np.asarray(expected)
    actual = np.asarray(actual)

    breakpoints = np.percentile(
        expected,
        np.linspace(0, 100, bins + 1)
    )

    breakpoints = np.unique(breakpoints)

    if len(breakpoints) < 3:
        return 0.0

    expected_counts = np.histogram(
        expected,
        bins=breakpoints
    )[0]

    actual_counts = np.histogram(
        actual,
        bins=breakpoints
    )[0]

    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)

    expected_pct = np.where(
        expected_pct == 0,
        0.0001,
        expected_pct
    )

    actual_pct = np.where(
        actual_pct == 0,
        0.0001,
        actual_pct
    )

    psi = np.sum(
        (actual_pct - expected_pct)
        * np.log(actual_pct / expected_pct)
    )

    return psi

drift_results = []

for column in X_train.columns:

    psi_value = calculate_psi(
        X_train[column].dropna(),
        X_test[column].dropna()
    )

    if psi_value < 0.10:
        status = "Low Drift"

    elif psi_value < 0.25:
        status = "Moderate Drift"

    else:
        status = "Significant Drift"

    drift_results.append({
        "feature": column,
        "PSI": round(psi_value, 4),
        "drift_status": status
    })

drift_report = pd.DataFrame(drift_results)

display(drift_report)

drift_report.to_csv(
    "data/audit/data_drift_report.csv",
    index=False
)

train_matrix = X_train_processed.copy()

train_matrix["target"] = y_train.values

train_matrix.to_csv(
    "data/processed/train_features.csv",
    index=False
)

test_matrix = X_test_processed.copy()

test_matrix["target"] = y_test.values

test_matrix.to_csv(
    "data/processed/test_features.csv",
    index=False
)

final_audit = []

for column in X_train_processed.columns:

    final_audit.append({

        "feature": column,

        "train_missing":
            int(X_train_processed[column].isnull().sum()),

        "test_missing":
            int(X_test_processed[column].isnull().sum()),

        "train_mean":
            round(X_train_processed[column].mean(), 4),

        "train_std":
            round(X_train_processed[column].std(), 4),

        "test_mean":
            round(X_test_processed[column].mean(), 4),

        "test_std":
            round(X_test_processed[column].std(), 4),

        "data_type":
            str(X_train_processed[column].dtype)

    })

final_quality_audit = pd.DataFrame(final_audit)

final_quality_audit.to_csv(
    "data/audit/final_data_quality_audit.csv",
    index=False
)

y_train.to_csv(
    "data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "data/processed/y_test.csv",
    index=False
)

print("\n" + "=" * 60)
print("DATA ENGINEER PIPELINE COMPLETE")
print("=" * 60)

print("\nFinal Training Matrix:")
print(X_train_processed.shape)

print("\nFinal Testing Matrix:")
print(X_test_processed.shape)

print("\nGenerated Files:")
print("""
data/
├── raw/
│
├── processed/
│   ├── train_features.csv
│   ├── test_features.csv
│   ├── y_train.csv
│   └── y_test.csv
│
└── audit/
    ├── source_info.json
    ├── schema_audit.csv
    ├── data_quality_before_cleaning.csv
    ├── invalid_records.csv
    ├── outlier_report.csv
    ├── data_drift_report.csv
    └── final_data_quality_audit.csv
""")

print("\nLeakage Prevention:")
print("""
✓ Train/test split performed BEFORE preprocessing
✓ Imputer fitted only on training data
✓ Outlier boundaries calculated only from training data
✓ Scaler fitted only on training data
✓ Test data only transformed using training parameters
""")

print("\nData Engineer pipeline successfully completed.")

Project directories created.
DATA ACQUISITION
Dataset successfully acquired.
Feature shape: (303, 13)
Target shape: (303, 1)

Source information saved.

Initial dataset shape: (303, 14)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  num  
0  0.0   6.0    0  
1  3.0   3.0    2  
2  2.0   7.0    1  
3  0.0   3.0    0  
4  0.0   3.0    0  

SCHEMA VALIDATION
Schema validation PASSED.
Expected columns found: 14

Data types:
      column    dtype  missing_count  unique_values
0        age    int64              0             41
1        sex    in

,column,dtype,rows,missing_count,missing_percentage,unique_values,duplicate_count
0,age,int64,303,0,0.00,41,262
1,sex,int64,303,0,0.00,2,301
2,cp,int64,303,0,0.00,4,299
3,trestbps,int64,303,0,0.00,50,253
4,chol,int64,303,0,0.00,152,151
5,fbs,int64,303,0,0.00,2,301
6,restecg,int64,303,0,0.00,3,300
7,thalach,int64,303,0,0.00,91,212
8,exang,int64,303,0,0.00,2,301
9,oldpeak,float64,303,0,0.00,40,263



DUPLICATE RECORD CHECK
Duplicate records: 0
No duplicate records found.

INVALID RECORD VALIDATION
     column  invalid_count
0       age              0
1  trestbps              0
2      chol              0
3   thalach              0
4       num              0
Invalid records removed: 0

TARGET TRANSFORMATION

Binary target distribution:
target
0    164
1    139
Name: count, dtype: int64

Target percentages:
target
0    54.13
1    45.87
Name: proportion, dtype: float64

Features: (303, 13)
Target: (303,)

TRAIN / TEST SPLIT
Training samples: 242
Testing samples: 61

Training target distribution:
 target
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Testing target distribution:
 target
0    0.540984
1    0.459016
Name: proportion, dtype: float64

OUTLIER DETECTION


,feature,Q1,Q3,IQR,lower_bound,upper_bound,train_outliers,test_outliers
0,age,48.00,61.00,13.00,28.500,80.500,0,0
1,sex,0.00,1.00,1.00,-1.500,2.500,0,0
2,cp,2.25,4.00,1.75,-0.375,6.625,0,0
3,trestbps,120.00,140.00,20.00,90.000,170.000,6,3
4,chol,212.00,277.75,65.75,113.375,376.375,5,0
5,fbs,0.00,0.00,0.00,0.000,0.000,35,10
6,restecg,0.00,2.00,2.00,-3.000,5.000,0,0
7,thalach,134.50,166.00,31.50,87.250,213.250,1,0
8,exang,0.00,1.00,1.00,-1.500,2.500,0,0
9,oldpeak,0.00,1.60,1.60,-2.400,4.000,4,1



BUILDING PREPROCESSING PIPELINE
Fitting preprocessing pipeline on TRAINING data...
Training preprocessing complete.
Transforming TEST data using training parameters...
Testing preprocessing complete.

POST-PROCESSING QUALITY CHECK
Training missing values: 0
Testing missing values: 0
Missing-value validation PASSED.

DATA DRIFT CHECK


,feature,PSI,drift_status
0,age,0.5365,Significant Drift
1,sex,0.0000,Low Drift
2,cp,0.0182,Low Drift
3,trestbps,0.2998,Significant Drift
4,chol,0.2776,Significant Drift
5,fbs,0.0000,Low Drift
6,restecg,0.0000,Low Drift
7,thalach,0.1028,Moderate Drift
8,exang,0.0000,Low Drift
9,oldpeak,0.1935,Moderate Drift



DATA ENGINEER PIPELINE COMPLETE

Final Training Matrix:
(242, 13)

Final Testing Matrix:
(61, 13)

Generated Files:

data/
├── raw/
│
├── processed/
│   ├── train_features.csv
│   ├── test_features.csv
│   ├── y_train.csv
│   └── y_test.csv
│
└── audit/
    ├── source_info.json
    ├── schema_audit.csv
    ├── data_quality_before_cleaning.csv
    ├── invalid_records.csv
    ├── outlier_report.csv
    ├── data_drift_report.csv
    └── final_data_quality_audit.csv


Leakage Prevention:

✓ Train/test split performed BEFORE preprocessing
✓ Imputer fitted only on training data
✓ Outlier boundaries calculated only from training data
✓ Scaler fitted only on training data
✓ Test data only transformed using training parameters


Data Engineer pipeline successfully completed.
